# DGX Training: U-Net Melting-Layer Detector

This notebook trains a U-Net to detect the melting layer (bright band) on GPM DPR data across ~98 files on a DGX GPU system. It uses mixed precision, supports multi-GPU with MirroredStrategy, and streams data via tf.data.

In [ ]:
# GPU and distribution strategy setup (mixed precision, memory growth, optional XLA)
import os, random, json, time
import numpy as np

# Optional: Enable XLA. Must be set before importing TensorFlow
os.environ.setdefault('TF_XLA_FLAGS', '--tf_xla_auto_jit=2')

import tensorflow as tf
from tensorflow.keras import mixed_precision

# Set global seeds for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# GPU memory growth
try:
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs visible: {len(gpus)} ->", gpus)
except Exception as e:
    print("GPU setup warning:", e)

# Mixed precision
mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision policy:", mixed_precision.global_policy())

# Strategy (multi-GPU ready)
strategy = tf.distribute.MirroredStrategy()
print("Strategy devices:", strategy.num_replicas_in_sync)

In [ ]:
# Configuration (paths, hyperparameters, mask mode)
import os
from pathlib import Path

# Data
DATA_DIR = 'GPM_DPR_2024'          # folder with ~98 HDF5 files
FILE_GLOB = '**/*.HDF5'
MAX_FILES = None                    # None -> use all

# Preprocessing / features
N_BIN_TARGET = 256                  # vertical bins after pad/crop
INCLUDE_LATLON = True               # add lat/lon channels (normalized)
USE_Z_FINAL = True                  # prefer /SLV/zFactorFinal
MASK_MODE = 'baseline'              # 'baseline'|'dataset_full'|'conservative'
USE_DFR_ML = True                   # prefer DFR ML bounds when available
QUALITY_MIN = 0                     # optional quality threshold for masks
MIN_VAL, MAX_VAL = -30.0, 60.0      # dBZ normalization bounds
SKIP_EMPTY_MASK = True

# Training
BATCH_SIZE = 8
EPOCHS = 15
LEARNING_RATE = 1e-3
INFER_THRESHOLD = 0.45

# Artifacts
CKPT_DIR = 'checkpoints'
LOG_DIR = 'logs/fit'
SAVEDMODEL_DIR = 'saved_model'
FIG_DIR = 'figures'
CONFIG_JSON = 'config.json'
SPLIT_JSON = 'split.json'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(SAVEDMODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print({
    'DATA_DIR': DATA_DIR,
    'MASK_MODE': MASK_MODE,
    'INCLUDE_LATLON': INCLUDE_LATLON,
    'N_BIN_TARGET': N_BIN_TARGET,
    'EPOCHS': EPOCHS,
    'BATCH_SIZE': BATCH_SIZE,
    'INFER_THRESHOLD': INFER_THRESHOLD,
})

In [ ]:
# Reload helper module to pick up latest fixes
import importlib
import dgx_ml as dgx

dgx = importlib.reload(dgx)
print("Reloaded dgx_ml:", dgx.__file__)

In [ ]:
# Discover files and create reproducible train/val/test split (80/10/10)
import json
import numpy as np
from pathlib import Path
import dgx_ml as dgx

all_files = dgx.list_hdf5_files(DATA_DIR, pattern=FILE_GLOB)
if MAX_FILES is not None:
    all_files = all_files[:MAX_FILES]
print(f"Discovered {len(all_files)} HDF5 files\nSample: {all_files[:3]}")
assert len(all_files) >= 98, "Expected at least 98 HDF5 files. Please check DATA_DIR and FILE_GLOB."

# Deterministic split
rng = np.random.default_rng(RANDOM_SEED)
idx = np.arange(len(all_files))
rng.shuffle(idx)

n_total = len(idx)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
train_idx = idx[:n_train]
val_idx = idx[n_train:n_train+n_val]
test_idx = idx[n_train+n_val:]

train_files = [all_files[i] for i in train_idx]
val_files = [all_files[i] for i in val_idx]
test_files = [all_files[i] for i in test_idx]

print(f"Split -> train: {len(train_files)}, val: {len(val_files)}, test: {len(test_files)}")

split_obj = {
    'seed': RANDOM_SEED,
    'train': train_files,
    'val': val_files,
    'test': test_files,
}
with open(SPLIT_JSON, 'w', encoding='utf-8') as f:
    json.dump(split_obj, f, indent=2)
print(f"Saved split to {SPLIT_JSON}")

In [ ]:
# Data loading and preprocessing utilities (wrapping dgx_ml)
import h5py
import numpy as np
import dgx_ml as dgx

INPUT_CHANNELS = 4 if INCLUDE_LATLON else 2

# Generator that yields (H, W, C) and (H, W, 1) per scan lazily from a list of files
def sample_generator(files,
                     mask_mode=MASK_MODE,
                     use_z_final=USE_Z_FINAL,
                     use_dfr_ml=USE_DFR_ML,
                     include_latlon=INCLUDE_LATLON,
                     nbin_target=N_BIN_TARGET,
                     min_val=MIN_VAL,
                     max_val=MAX_VAL,
                     skip_empty=SKIP_EMPTY_MASK):
    for fp in files:
        # Build arrays for one file at a time to bound memory
        X, Y = dgx.build_arrays_for_mask_mode([fp],
                                              mask_mode=mask_mode,
                                              use_z_final=use_z_final,
                                              use_dfr_ml=use_dfr_ml,
                                              include_latlon=include_latlon,
                                              nbin_target=nbin_target,
                                              min_val=min_val,
                                              max_val=max_val,
                                              max_scans=None)
        for i in range(X.shape[0]):
            if skip_empty and np.sum(Y[i]) == 0:
                continue
            yield X[i].astype(np.float32), Y[i].astype(np.float32)

print("Generator ready. Example probe on first train file (no execution):", os.path.basename(train_files[0]))

In [ ]:
# tf.data pipelines: interleave files, shuffle/cache/batch/prefetch
AUTOTUNE = tf.data.AUTOTUNE

output_signature = (
    tf.TensorSpec(shape=(N_BIN_TARGET, None, INPUT_CHANNELS), dtype=tf.float32),
    tf.TensorSpec(shape=(N_BIN_TARGET, None, 1), dtype=tf.float32)
)

def make_dataset(file_list, shuffle=False):
    def gen():
        yield from sample_generator(file_list)
    ds = tf.data.Dataset.from_generator(gen, output_signature=output_signature)
    if shuffle:
        ds = ds.shuffle(8192, seed=RANDOM_SEED, reshuffle_each_iteration=True)
    # Optional: cache to memory; on DGX with NVMe you can use .cache('/nvme/cache.tfd')
    ds = ds.batch(BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, shuffle=True)
val_ds = make_dataset(val_files, shuffle=False)
print(train_ds, val_ds)

In [ ]:
# Model definition and compile (inside strategy.scope)
with strategy.scope():
    model = dgx.build_unet_2d(input_channels=INPUT_CHANNELS, nbin_target=N_BIN_TARGET, base_filters=16)
    # Choose loss: focal for imbalance, else BCE
    use_focal = True
    if use_focal:
        loss_fn = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0)
    else:
        loss_fn = tf.keras.losses.BinaryCrossentropy()
    opt = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(optimizer=opt, loss=loss_fn, metrics=['accuracy'])

model.summary(line_length=120)
print("Input shape ->", (N_BIN_TARGET, None, INPUT_CHANNELS))

In [ ]:
# Train with callbacks: EarlyStopping, ModelCheckpoint, TensorBoard
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, TensorBoard
run_name = f"unet_{MASK_MODE}_latlon{int(INCLUDE_LATLON)}_nbin{N_BIN_TARGET}"
ckpt_path = os.path.join(CKPT_DIR, run_name + "_best.h5")
log_dir = os.path.join(LOG_DIR, time.strftime("%Y%m%d-%H%M%S") + "_" + run_name)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(ckpt_path, monitor='val_loss', save_best_only=True, save_weights_only=False),
    TensorBoard(log_dir=log_dir)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("Best checkpoint:", ckpt_path)

In [ ]:
# Save artifacts: SavedModel, .h5, config JSON, split JSON
best_h5 = ckpt_path
saved_model_path = os.path.join(SAVEDMODEL_DIR, run_name)

# Save current model state as SavedModel and as .h5
model.save(saved_model_path)  # SavedModel
final_h5 = os.path.join(SAVEDMODEL_DIR, run_name + ".h5")
model.save(final_h5)
print("Saved:", saved_model_path, "and", final_h5)

config = {
    'DATA_DIR': DATA_DIR,
    'FILE_GLOB': FILE_GLOB,
    'MASK_MODE': MASK_MODE,
    'USE_DFR_ML': USE_DFR_ML,
    'INCLUDE_LATLON': INCLUDE_LATLON,
    'USE_Z_FINAL': USE_Z_FINAL,
    'N_BIN_TARGET': N_BIN_TARGET,
    'MIN_VAL': MIN_VAL,
    'MAX_VAL': MAX_VAL,
    'BATCH_SIZE': BATCH_SIZE,
    'EPOCHS': EPOCHS,
    'LEARNING_RATE': LEARNING_RATE,
    'INFER_THRESHOLD': INFER_THRESHOLD,
    'RUN_NAME': run_name,
    'BEST_H5': best_h5,
    'SAVEDMODEL_PATH': saved_model_path,
}
with open(CONFIG_JSON, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
print(f"Wrote {CONFIG_JSON}")

# split.json was already written; ensure it's present
assert os.path.exists(SPLIT_JSON), "split.json missing"
print("Artifacts saved.")

In [ ]:
# Quick validation visualization (paper-style)
import matplotlib.pyplot as plt

# Build a small array from one validation file for qualitative inspection
val_probe_file = val_files[0]
Xv, Yv = dgx.build_arrays_for_mask_mode([val_probe_file],
                                        mask_mode=MASK_MODE,
                                        use_z_final=USE_Z_FINAL,
                                        use_dfr_ml=USE_DFR_ML,
                                        include_latlon=INCLUDE_LATLON,
                                        nbin_target=N_BIN_TARGET,
                                        min_val=MIN_VAL,
                                        max_val=MAX_VAL,
                                        max_scans=32)

# Predict
Yp = model.predict(Xv, verbose=0)
if Yp.ndim == 4 and Yp.shape[-1] == 1:
    Yp = Yp[..., 0]
Yt = Yv[..., 0]

thr = INFER_THRESHOLD
idx = 0  # first sample
pred = Yp[idx]
pred_bin = (pred >= thr).astype(np.uint8)

# Denormalize reflectivity for display (Ku channel assumed at index 0)
ku_dbz = Xv[idx, ..., 0] * (MAX_VAL - MIN_VAL) + MIN_VAL

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
im0 = axs[0].imshow(ku_dbz, origin='lower', aspect='auto', cmap='RdYlGn_r', vmin=MIN_VAL, vmax=MAX_VAL)
axs[0].set_title('Ku Reflectivity (dBZ)'); plt.colorbar(im0, ax=axs[0])
im1 = axs[1].imshow(pred, origin='lower', aspect='auto', cmap='hot', vmin=0, vmax=1)
axs[1].set_title(f'Prediction (max={pred.max():.2f})'); plt.colorbar(im1, ax=axs[1])
im2 = axs[2].imshow(pred_bin, origin='lower', aspect='auto', cmap='gray_r', vmin=0, vmax=1)
axs[2].set_title(f'Binary ≥ {thr:.2f}'); plt.colorbar(im2, ax=axs[2])
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'dgx_val_quick.png')
plt.savefig(fig_path, dpi=200); plt.show()
print('Saved figure to', fig_path)